In [15]:
import pandas as pd
import numpy as np
import duckdb
import lightgbm as lgb
from datetime import datetime
from pandas.tseries.offsets import MonthEnd
import gc
import glob
import os

In [16]:
def gather_data(start_date_1,end_date_1,start_date_2=None,end_date_2=None):
    if start_date_2==None:
        start_date_2=start_date_1
    if end_date_2==None:
        end_date_2=end_date_1

    #query the features parquet for rows with desired dates
    query = """
    SELECT *
    FROM 'zone_features.parquet'
    WHERE (
        (date >= ? AND date <= ?)
        OR
        (date >= ? AND date <= ?)
    )
    ORDER BY date, he
    """

    return con.execute(
        query,
        [start_date_1, end_date_1, start_date_2, end_date_2]
    ).df()

In [17]:
def train_model(train_data,feature_cols):

    #prepare training data with intended feature_columns
    X_train = train_data[feature_cols]

    # designate categorical features
    cols=X_train.columns
    if "bus_unique_id" in cols:
        X_train["bus_unique_id"] = X_train["bus_unique_id"].astype("category")
    if "zone_name" in cols:
        X_train["zone_name"] = X_train["zone_name"].astype("category")

    #prepare actual values for training
    y_train = train_data["pd"]

    #train the model on the data
    model = lgb.LGBMRegressor(force_row_wise=True,verbosity=-1)
    trained_model = model.fit(X_train,y_train)
    
    return trained_model

In [18]:
def predict_demand(model,prediction_day_data,feature_cols,model_name,predicted_on):

    #prepare future data with intended feature_columns
    X_test=prediction_day_data[feature_cols]

    # designate categorical features
    cols=X_test.columns
    if "bus_unique_id" in cols:
        X_test["bus_unique_id"] = X_test["bus_unique_id"].astype("category")
    if "zone_name" in cols:
        X_test["zone_name"] = X_test["zone_name"].astype("category")
    
    #get predictions for future data
    results=model.predict(X_test)
    
    return pd.DataFrame({
        "model_name" : model_name,
        "forecast_created_at" : predicted_on,
        "date" : prediction_day_data["date"],
        "he" : prediction_day_data["he"],
        "zone_name" : prediction_day_data["zone_name"],
        "predict_pd" : results,
        "actual_pd":prediction_day_data["pd"],
        "baseline_pd":prediction_day_data["same_hour_prev_year"]
    })

In [19]:
def calculate_statistics(df,month):
        #calculate statistics on predictions and baseline
        MAE_predict =(df["actual_pd"] - df["predict_pd"]).abs().mean()
        RMSE_predict = (((df["actual_pd"] - df["predict_pd"]) ** 2).mean()) ** 0.5
        WMAPE_predict = ((df["actual_pd"] - df["predict_pd"]).abs().sum() / df["actual_pd"].sum())
        MAE_baseline =(df["actual_pd"] - df["baseline_pd"]).abs().mean()
        RMSE_baseline = (((df["actual_pd"] - df["baseline_pd"]) ** 2).mean()) ** 0.5
        WMAPE_baseline = ((df["actual_pd"] - df["baseline_pd"]).abs().sum() / df["actual_pd"].sum())
        statistics={"month":month,
                    "MAE_predict":MAE_predict,
                    "RMSE_predict":RMSE_predict,
                    "WMAPE_predict":WMAPE_predict,
                    "MAE_baseline":MAE_baseline,
                    "RMSE_baseline":RMSE_baseline,
                    "WMAPE_baseline":WMAPE_baseline
        }
        return statistics


In [20]:
def find_bus_share_loads(df):
    df["zone_name"] = df["zone_name"].astype("string")
    df["model_name"] = df["model_name"].astype("string")
    
    con.register("zone_predictions", df)
    
    bus_df = con.execute(f"""

        WITH shares AS (SELECT * FROM 'lagged_bus_share.parquet' WHERE target_year = {year} AND target_month = {month}),
        
        bus_predictions AS (
            SELECT
                z.model_name,
                z.forecast_created_at,
                z.date AS target_date,
                z.he AS he,
                s.bus_id,
                s.zone_name AS zone_id,
                z.predict_pd * s.avg_bus_share AS predict_pd
            FROM zone_predictions z
            JOIN shares s
                ON z.zone_name = s.zone_name
               AND z.he = s.he
        )
        
        SELECT
            p.model_name,
            p.forecast_created_at,
            p.target_date,
            p.he,
            p.bus_id,
            p.zone_id,
            p.predict_pd,
            f.pd AS actual_pd,
            f.same_hour_prev_year AS baseline_pd
        FROM bus_predictions AS p
        
        LEFT JOIN 'features.parquet' AS f
            ON p.bus_id = f.bus_unique_id
           AND p.target_date = f.date
           AND p.he = f.he
        
        """).df()
    return bus_df

In [21]:
con=duckdb.connect()

In [22]:
dates = pd.date_range(
    start="2022-01-01",
    end="2025-12-31",
    freq="D"
)
first_day_2025_idx=dates.get_loc('2025-01-01')

In [23]:
next_day_feature_cols = [
"zone_name",

"he",
"dow",
"month",
"day_of_year",
"is_weekend",

"same_hour_prev_week",
"same_hour_prev_35day",
"same_hour_prev_year",
    
]
monthly_feature_cols = next_day_feature_cols
monthly_feature_cols.remove("same_hour_prev_week")

In [24]:
#loop initialization variables
daily_retrain_frequency=7
train_data_lags={"start_1" : 37,
                 "end_1" : 2,
}
monthly_train_data_lags={"start_1" : 37,
                         "end_1" : 2,
}

In [25]:
days_until_retrain=0#reset the retraining countdown to train the next_day model on the first day

#initialize a dataframe for the required statistics
next_day_statistics=pd.DataFrame(columns=["month","MAE_predict","RMSE_predict","WMAPE_predict","MAE_baseline","RMSE_baseline","WMAPE_baseline"])
next_month_statistics=pd.DataFrame(columns=["month","MAE_predict","RMSE_predict","WMAPE_predict","MAE_baseline","RMSE_baseline","WMAPE_baseline"])


#Loop through every day of the test period --------------------------------------------------------------------------------------------------------------------------------------------------------------------
for i in range(first_day_2025_idx,len(dates)):
    
    #get time based variables
    prediction_day=dates[i]
    previous_day=dates[i-1]
    week=prediction_day.week
    month=prediction_day.month
    year=prediction_day.year
    first_day_month = pd.Timestamp(year=year, month=month, day=1)
    last_day_month = first_day_month + MonthEnd(0)

    
#Gather future data for the whole month on the first day of the month------------------------------------------------------------------------------------------------------------------------------------------
    if prediction_day==first_day_month:
        future_data=gather_data(first_day_month,last_day_month)
        
        
#Tain monthly model----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------    
    if prediction_day==first_day_month:
        print(f"\rProcessing month {month}/12", end="")
        
        train_data=gather_data(dates[i-monthly_train_data_lags["start_1"]],dates[i-monthly_train_data_lags["end_1"]]) #gather the training data
        
        next_month_model=train_model(train_data,monthly_feature_cols) #train the next month forcast model
        
        monthly_predicted_on=previous_day #save the last day of the last month as the forcast date
        
        del train_data #delete train_data variable
        gc.collect

        #reset the monthly and daily model predictions for the new month
        monthly_predictions=[]
        next_day_predictions=[]

#Tain daily model----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------        
    if days_until_retrain==0:
        days_until_retrain=daily_retrain_frequency#reset the retrain countdown
        
        train_data=gather_data(dates[i-train_data_lags["start_1"]],dates[i-train_data_lags["end_1"]])#gather the training data
        
        next_day_model=train_model(train_data,next_day_feature_cols)#train the next day forcast model
        
        del train_data #delete train_data variable
        gc.collect
        
    days_until_retrain-=1 #decrease the retrain countdown for the next day model

    
#Make Predictions for prediction_day----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------       
    prediction_day_data=future_data[future_data["date"]==prediction_day] #get the data for the prediction day
    
    if not prediction_day_data.empty:
        #get predictions from both models
        next_day_model_predictions=predict_demand(next_day_model,prediction_day_data,next_day_feature_cols,'next-day model',previous_day)
        monthly_model_predictions=predict_demand(next_month_model,prediction_day_data,monthly_feature_cols,'next-month model',monthly_predicted_on)

        #append the predictions to this months list of predictions
        next_day_predictions.append(next_day_model_predictions)
        monthly_predictions.append(monthly_model_predictions)

#Export predictions to a parquet file at the end of every mont--------------------------------------------------------------------------------------------------------------------------------------------------------       
    if prediction_day==last_day_month:
        daily_df=pd.concat(next_day_predictions,ignore_index=True) #concatenate the predictions for the month
        daily_df["predict_pd"] = daily_df["predict_pd"].clip(lower=0) #correct negative demand predictions
        
        daily_bus_df = find_bus_share_loads(daily_df) #convert the zone loads into bus loads using lagged bus share
        
        daily_bus_df.to_parquet(f"temporary_zone_daily_predictions_{year}_{month:02d}.parquet",engine="pyarrow",index=False) #export predictions to a parquet
        
        del next_day_predictions #delete predictions list for the month
        gc.collect

#Get monthly bus level predictions-----------------------------------------------------------------------------------------------------------------------------------------------
        monthly_df=pd.concat(monthly_predictions,ignore_index=True) #concatenate the predictions for the month
        monthly_df["predict_pd"] = monthly_df["predict_pd"].clip(lower=0) #correct negative demand predictions
        
        monthly_bus_df = find_bus_share_loads(monthly_df) #convert the zone loads into bus loads using lagged bus share
        
        monthly_bus_df.to_parquet(f"temporary_zone_monthly_predictions_{year}_{month:02d}.parquet",engine="pyarrow",index=False) #export predictions to a parquet
        
        del monthly_predictions #delete predictions list for the month
        gc.collect

#Append statistics for the month to the respective statistics dataframe---------------------------------------------------------------------------------------------------------------------------------------------
        next_day_statistics.loc[len(next_day_statistics)]=calculate_statistics(daily_bus_df,month)
        next_month_statistics.loc[len(next_month_statistics)]=calculate_statistics(monthly_bus_df,month)
        
print(f"\rDone                  ", end="")

Done                  

In [26]:
#combine all prediction parquets into a single file
try:
    con.execute("""
    
    COPY (
        SELECT * EXCLUDE (actual_pd, baseline_pd)
        FROM 'temporary_zone_daily_predictions*.parquet'
        UNION ALL
        SELECT * EXCLUDE (actual_pd, baseline_pd)
        FROM 'temporary_zone_monthly_predictions*.parquet'
    )
    
    TO 'zone_bus_share_predictions.parquet'
    (FORMAT PARQUET)
    
    """)
    #remove temporary prediction files 
    for file in glob.glob("temporary_zone_daily_predictions*.parquet"):
        os.remove(file)
    for file in glob.glob("temporary_zone_monthly_predictions*.parquet"):
        os.remove(file)
except duckdb.IOException:
    print("Predictions can not be combined")

In [35]:
next_day_statistics

,month,MAE_predict,RMSE_predict,WMAPE_predict,MAE_baseline,RMSE_baseline,WMAPE_baseline
0,1,3.202493,9.657744,0.279384,3.827095,11.055613,0.317327
1,2,2.911154,8.183439,0.271510,3.515136,11.825625,0.311670
2,3,1.514559,4.699234,0.155200,2.912370,10.163399,0.282347
3,4,2.333079,6.796741,0.219831,3.188529,12.447486,0.283182
4,5,2.351857,6.405204,0.203425,3.552337,13.446868,0.291842
5,6,3.042837,10.299496,0.232439,3.530580,12.479792,0.257579
6,7,3.231932,9.709095,0.242310,3.753289,12.998214,0.268070
7,8,2.454174,6.954441,0.177981,3.424345,13.092314,0.236521
8,9,1.567791,4.455053,0.124028,3.104410,12.314392,0.232654
9,10,1.350954,4.019025,0.119156,3.077920,10.248042,0.256455


In [36]:
next_month_statistics

,month,MAE_predict,RMSE_predict,WMAPE_predict,MAE_baseline,RMSE_baseline,WMAPE_baseline
0,1,3.307801,10.473479,0.288571,3.827095,11.055613,0.317327
1,2,2.633725,8.054439,0.245636,3.515136,11.825625,0.311670
2,3,1.424294,4.404418,0.145950,2.912370,10.163398,0.282347
3,4,2.293119,6.740456,0.216066,3.188529,12.447486,0.283182
4,5,2.265065,6.148901,0.195918,3.552337,13.446868,0.291842
5,6,3.193350,9.102463,0.243936,3.530580,12.479792,0.257579
6,7,3.273732,9.332596,0.245444,3.753289,12.998214,0.268070
7,8,2.437118,6.921009,0.176744,3.424345,13.092314,0.236522
8,9,1.802427,4.776677,0.142590,3.104410,12.314392,0.232654
9,10,1.390526,4.066648,0.122646,3.077920,10.248042,0.256455
